In [ ]:
!pip install pandas supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 437.0 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 412.5 kB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
from supabase import create_client, Client
import os

# ==========================================
# Configuration Variables
# ==========================================
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
CSV_FILENAME = "tournament_players.csv"
TABLE_NAME = "trainers" # Adjust this if your target table is named differently

def fetch_all_db_records(supabase: Client, table: str) -> list:
    """Fetches all existing full records from Supabase."""
    print("Fetching full existing database records...")
    all_records = []
    limit = 1000
    offset = 0
    
    while True:
        # Fetching '*' gets all columns, satisfying NOT NULL constraints later
        response = supabase.table(table).select("*").range(offset, offset + limit - 1).execute()
        data = response.data
        
        if not data: break
        all_records.extend(data)
        if len(data) < limit: break
        
        offset += limit
        
    print(f"Fetched {len(all_records)} complete records from the database.")
    return all_records

def read_modify_write_update(csv_path: str, url: str, key: str, table: str):
    supabase: Client = create_client(url, key)
    
    # 1. READ: Fetch all existing database rows
    db_records = fetch_all_db_records(supabase, table)
    
    # 2. Load and prep the CSV data
    print(f"\nProcessing {csv_path}...")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: Could not find '{csv_path}'.")
        return

    # Deduplicate keeping the latest entry
    df_deduped = df.drop_duplicates(subset=['Dropdown Text'], keep='last').copy()
    
    # Create a fast lookup dictionary: {'id': 'alphabet_name'}
    # e.g., {'ph00497299': 'Melvyn Germono', ...}
    csv_updates_lookup = dict(zip(df_deduped['Dropdown Text'], df_deduped['Player Name']))
    
    # 3. MODIFY: Update the database records in-memory
    records_to_upsert = []
    
    for row in db_records:
        row_id = row['id']
        
        # If this database ID exists in our CSV, update it
        if row_id in csv_updates_lookup:
            row['alphabet_name'] = csv_updates_lookup[row_id]
            # Add the fully-formed, modified row to our upsert payload
            records_to_upsert.append(row)
            
    # 4. WRITE: Bulk Upsert the modified records
    if not records_to_upsert:
        print("No matching existing records found to update.")
        return
        
    print(f"\nExecuting bulk upsert for {len(records_to_upsert)} modified records...")
    try:
        response = supabase.table(table).upsert(records_to_upsert).execute()
        print("Bulk update completed successfully!")
        
        return pd.DataFrame(response.data).head()
        
    except Exception as e:
        print(f"An error occurred during the Supabase operation: {e}")

# ==========================================
# Execution
# ==========================================
preview = read_modify_write_update(
    csv_path=CSV_FILENAME, 
    url=SUPABASE_URL, 
    key=SUPABASE_KEY, 
    table=TABLE_NAME
)

preview

Fetching full existing database records...
Fetched 1687 complete records from the database.

Processing tournament_players.csv...

Executing bulk upsert for 574 modified records...
Bulk update completed successfully!


,id,nickname,full_name,alphabet_name,division,user_id,is_deleted,created_at,updated_at
0,ph42401421,Michael,None,Michael Aninang,master,None,False,2026-02-24T11:58:52.89304+00:00,2026-04-05T11:13:11.105599+00:00
1,ph64096588,cen,None,Abero Isaiah Doctor Geronga,master,None,False,2026-02-24T11:58:52.89304+00:00,2026-04-05T11:13:11.105599+00:00
2,ph95280670,Sky,None,Christian Sky T Mendoza,master,None,False,2026-02-24T11:58:52.89304+00:00,2026-04-05T11:13:11.105599+00:00
3,ph56189230,Pogi,None,El Mendoza,master,None,False,2026-02-24T11:58:52.89304+00:00,2026-04-05T11:13:11.105599+00:00
4,ph69917369,Lukaruio,None,Karu,master,None,False,2026-02-24T11:58:52.89304+00:00,2026-04-05T11:13:11.105599+00:00
